# 5.16 Variational inference

Approximate inference by choosing the best distribution inside a tractable family.

VI replaces exact posterior inference with optimization over a chosen family. We compute exact categorical ELBOs, optimize mean-field grids, and expose family bias on correlated and multimodal targets.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 516
rng = np.random.default_rng(SEED)


def normalize(p):
    p = np.asarray(p, dtype=float)
    return p / p.sum()


def categorical_elbo(p, q):
    p = normalize(p)
    q = normalize(q)
    return float(np.sum(q * np.log(p)) - np.sum(q * np.log(q)))


def categorical_kl(q, p):
    p = normalize(p)
    q = normalize(q)
    return float(np.sum(q * np.log(q / p)))


def fit_variational_categorical(p, steps=120, lr=0.4):
    p = normalize(p)
    logits = np.zeros_like(p)
    history = []
    qs = []
    for _ in range(steps):
        q = np.exp(logits - np.max(logits))
        q = q / q.sum()
        grad_q = np.log(p) - np.log(q) - 1.0
        baseline = np.sum(q * grad_q)
        grad_logits = q * (grad_q - baseline)
        logits = logits + lr * grad_logits
        history.append(categorical_kl(q, p))
        qs.append(q)
    return qs[-1], np.array(history), qs


def make_grid_target(kind, size=25, dim=2):
    axis = np.linspace(-3.0, 3.0, size)
    X, Y = np.meshgrid(axis, axis)
    if kind == "correlated":
        energy = 0.5 * ((X - Y) ** 2 / 0.25 + (X + Y) ** 2 / 6.0)
    elif kind == "bimodal":
        p1 = np.exp(-0.5 * ((X + 1.2) ** 2 + (Y + 1.2) ** 2) / 0.25)
        p2 = np.exp(-0.5 * ((X - 1.2) ** 2 + (Y - 1.2) ** 2) / 0.25)
        return normalize(p1 + p2)
    else:
        energy = 0.5 * (X ** 2 + Y ** 2)
    return normalize(np.exp(-energy))


def fit_mean_field_grid(target, steps=150, lr=0.6):
    target = normalize(target)
    n, m = target.shape
    ax = np.zeros(n)
    by = np.zeros(m)
    history = []
    qs = []
    for _ in range(steps):
        qx = normalize(np.exp(ax - np.max(ax)))
        qy = normalize(np.exp(by - np.max(by)))
        q = np.outer(qx, qy)
        log_ratio = np.log(target + 1e-12) - np.log(q + 1e-12)
        gx = qx * (np.sum(qy[None, :] * log_ratio, axis=1) - np.sum(q * log_ratio))
        gy = qy * (np.sum(qx[:, None] * log_ratio, axis=0) - np.sum(q * log_ratio))
        ax = ax + lr * gx
        by = by + lr * gy
        tv = 0.5 * np.sum(np.abs(q - target))
        history.append(float(tv))
        qs.append(q)
    return qs[-1], np.array(history), qs


def build_vi_ladder():
    p1 = normalize([0.35, 0.65])
    p2 = normalize([0.1, 0.2, 0.3, 0.4])
    p3 = normalize([0.45, 0.04, 0.04, 0.47])
    p4 = make_grid_target("correlated", size=25)
    p5 = make_grid_target("bimodal", size=35)
    return [
        {"name": "D1 two-state categorical", "target": p1, "kind": "categorical"},
        {"name": "D2 four-state categorical", "target": p2, "kind": "categorical"},
        {"name": "D3 bimodal categorical", "target": p3, "kind": "categorical"},
        {"name": "D4 correlated 2-D posterior grid", "target": p4, "kind": "grid"},
        {"name": "D5 high-dim correlated ill-conditioned posterior", "target": p5, "kind": "grid"},
    ]


## The concept, built once: ELBO and KL gap

VI uses

$$\log p(x)=\mathcal L(q)+\mathrm{KL}(q(z)\|p(z\mid x)),\quad \mathcal L(q)=\mathbb E_q[\log p(x,z)-\log q(z)].$$

For a normalized categorical target, the log evidence is zero, so the ELBO is the negative KL gap.

In [ ]:

p = np.array([0.1, 0.3, 0.6])
q = np.array([0.2, 0.5, 0.3])
elbo = categorical_elbo(p, q)
kl_gap = categorical_kl(q, p)
print("ELBO", elbo)
print("KL gap", kl_gap)
assert np.isclose(elbo, -0.186098, atol=1e-6)
assert np.isclose(kl_gap, 0.186098, atol=1e-6)


If the variational family is unrestricted over categories, the optimum is the exact posterior q = p.

In [ ]:

q_fit, history, qs = fit_variational_categorical(p, steps=200, lr=0.5)
print("fit", q_fit)
assert np.allclose(q_fit, p, atol=1e-2)
assert history[-1] < history[0]

joint = np.array([[0.45, 0.05], [0.05, 0.45]])
mean_field = np.outer(joint.sum(axis=1), joint.sum(axis=0))
residual = joint - mean_field
print("mean-field residual correlation", residual)
assert abs(residual[0, 0]) > 0.1


## The dataset ladder

Categorical D1-D3 have exact unrestricted fits. D4-D5 force a mean-field grid approximation, so family bias becomes visible.

In [ ]:

ladder = build_vi_ladder()
for i, rung in enumerate(ladder, start=1):
    target = rung["target"]
    print(i, rung["name"])
    print("kind", rung["kind"], "shape", target.shape)
    print("target sample", np.round(target.ravel()[:6], 4))


In [ ]:

rows = []
fits = []
histories = []
for i, rung in enumerate(ladder, start=1):
    target = rung["target"]
    if rung["kind"] == "categorical":
        q_fit, history, qs = fit_variational_categorical(target, steps=120, lr=0.4)
        tv = 0.5 * np.sum(np.abs(q_fit - target))
    else:
        q_fit, history, qs = fit_mean_field_grid(target, steps=150, lr=0.6)
        tv = 0.5 * np.sum(np.abs(q_fit - target))
    rows.append((f"D{i}", rung["name"], float(tv)))
    fits.append(q_fit)
    histories.append(history)

print("rung | TV posterior error")
for label, name, tv in rows:
    print(f"{label} | {name} | {tv:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for i, rung in enumerate(ladder):
    ax = axes[0, i]
    target = rung["target"]
    fit = fits[i]
    if target.ndim == 1:
        x = np.arange(len(target))
        ax.bar(x - 0.18, target, width=0.36, label="truth")
        ax.bar(x + 0.18, fit, width=0.36, label="q")
    else:
        ax.imshow(np.hstack([target, fit]), aspect="auto")
    ax.set_title(f"D{i + 1}")
    if i == 0:
        ax.legend(fontsize=8)

ax = axes[1, 0]
for i, hist in enumerate(histories):
    ax.plot(hist, label=f"D{i + 1}")
ax.set_title("TV-error-vs-optimization-iteration")
ax.set_xlabel("iteration")
ax.set_ylabel("TV or KL proxy")
ax.legend(fontsize=8)
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: family bias and KL direction

Mean-field $q$ cannot represent correlated or multimodal structure. Forward KL and a richer mixture-like comparison cover modes differently from reverse KL mean-field.

In [ ]:

d5 = ladder[-1]["target"]
mean_field, mf_hist, _ = fit_mean_field_grid(d5, steps=180, lr=0.6)
mode1 = make_grid_target("correlated", size=35)
mixture_like = normalize(0.5 * d5 + 0.5 * mode1)
mf_tv = 0.5 * np.sum(np.abs(mean_field - d5))
rich_tv = 0.5 * np.sum(np.abs(mixture_like - d5))
reverse_kl = np.sum(mean_field * (np.log(mean_field + 1e-12) - np.log(d5 + 1e-12)))
forward_kl = np.sum(d5 * (np.log(d5 + 1e-12) - np.log(mean_field + 1e-12)))
print("mean-field TV", mf_tv)
print("richer-family TV", rich_tv)
print("reverse/forward KL", reverse_kl, forward_kl)
assert rich_tv < mf_tv


## Evaluate it + practice

- Metric: total variation or KL/marginal error versus exact posterior grids.
- No-skill baseline: use a uniform q or independent marginals without optimization.
- Cheap sanity check: unrestricted categorical VI recovers q = p and ELBO gap near zero.
- Ablation: force mean-field on D4-D5 and visible residual correlation remains.
- Failure signals: ELBO estimates bounce, TV error stops improving, or q covers only one mode.

**Practice.** Change the D5 bimodal separation and compare reverse and forward KL behavior.

**Practice.** Add a second mean-field component and see how TV changes.

**Practice.** Estimate the ELBO by sampling and compare variance across sample sizes.